In [ ]:
%pip install -U pip setuptools wheel
%pip install -U spacy
!python -m spacy download fr_dep_news_trf

%pip install stanza

%pip install spacy-lefff

# %pip install datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.7/397.7 MB 37.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('fr_dep_news_trf')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 48.7 MB/s eta 0:00:00
  Created wheel for msgpack: filename=msgpack-0.5.6-cp311-cp311-linux_x86_64.whl size=13917 sha256=276b3b49ae46f031a98e09074cb185bd6ad166d72e805f160e6bd49259bb7c91
  Stored in directory: /root/.cache/pip/wheels/7e/dc/61/86a33fcce57848e86e2d3dc6b03deaaefb34622feeb312f522
Successfully built msgpack
  Attempting uninstall: msgpack
    Found existing installation: msgpack 1.1.0
    Unin

In [ ]:
import pandas as pd
import json

import spacy
# from datasets import load_dataset
import stanza

In [ ]:
nlp = spacy.load("fr_dep_news_trf")
import fr_dep_news_trf
nlp = fr_dep_news_trf.load()

from spacy_lefff import LefffLemmatizer, POSTagger
from spacy.language import Language

@Language.factory('french_lemmatizer')
def create_french_lemmatizer(nlp, name):
    return LefffLemmatizer(after_melt=True, default=True)

@Language.factory('melt_tagger')
def create_melt_tagger(nlp, name):
    return POSTagger()

nlpl = spacy.load('fr_dep_news_trf')
nlpl.add_pipe('melt_tagger', after='parser')
nlpl.add_pipe('french_lemmatizer', after='melt_tagger')

# ds = load_dataset("maximoss/sick-fr-mt", split="test")

stanza.download('fr') # download French model
nlps = stanza.Pipeline('fr') # initialize French neural pipeline
# nlps = stanza.Pipeline('fr', tokenize_pretokenized=True) # initialize French neural pipeline with text already tokenised

INFO:stanza:Downloaded file to /root/stanza_resources/resources.json
INFO:stanza:Downloading default packages for language: fr (French) ...


INFO:stanza:Downloaded file to /root/stanza_resources/fr/default.zip
INFO:stanza:Finished downloading models and saved to /root/stanza_resources
INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/stanza_resources/resources.json
INFO:stanza:Loading these models for language: fr (French):
| Processor | Package            |
----------------------------------
| tokenize  | combined           |
| mwt       | combined           |
| pos       | combined_charlm    |
| lemma     | combined_nocharlm  |
| depparse  | combined_charlm    |
| ner       | wikinergold_charlm |

INFO:stanza:Using device: cpu
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: mwt
INFO:stanza:Loading: pos
INFO:stanza:Loading: lemma
INFO:stanza:Loading: depparse
INFO:stanza:Loading: ner
INFO:stanza:Done loading processors!


In [ ]:
with open('rte3_test_postags_lemmas_spacy.jsonl', 'w', encoding='utf-8') as f, open('rte3_test_input.txt', 'r', encoding='utf-8') as ds, open('rte3_test_input.txt', 'r', encoding='utf-8') as dfil:
  initial_sentences = dfil.readlines()
  for index, line in enumerate(ds):
    premise = nlp(line.strip())
    result=[[w.text, w.lemma_, w.pos_, w.tag_, w.dep_] for w in premise if w.text!="'"]
    json.dump({'id': (index+1), 'sentence': initial_sentences[index].strip(), 'pos_lemma': result}, f)
    f.write("\n")

with open('rte3_test_postags_lemmas_lefff.jsonl', 'w', encoding='utf-8') as f, open('rte3_test_input.txt', 'r', encoding='utf-8') as ds, open('rte3_test_input.txt', 'r', encoding='utf-8') as dfil:
  initial_sentences = dfil.readlines()
  for index, line in enumerate(ds):
    sent_lefff = nlpl(line.strip())
    result=[[w.text, w._.lefff_lemma, w._.melt_tagger] for w in sent_lefff if w.text!="'"]
    json.dump({'id': (index+1), 'sentence': initial_sentences[index].strip(), 'pos_lemma': result}, f)
    f.write("\n")

with open('rte3_test_postags_lemmas_stanza.jsonl', 'w', encoding='utf-8') as f, open('rte3_test_input.txt', 'r', encoding='utf-8') as ds, open('rte3_test_input.txt', 'r', encoding='utf-8') as dfil:
  initial_sentences = dfil.readlines()
  for index, line in enumerate(ds):
    sent_stanza = nlps(line.strip())
    result_stanza=[[w.text, w.lemma, w.pos, w.feats, w.deprel] for sent in sent_stanza.sentences for w in sent.words if w.text!="'"]
    json.dump({'id': (index+1), 'sentence': initial_sentences[index].strip(), 'pos_lemma': result_stanza}, f)
    f.write("\n")